# ins_gbm Example Usage

This notebook walks through the main `ins_gbm` workflow end to end: data loading, missing-value preparation, splitting, encoding, dimensionality reduction, model fitting, tuning, feature selection, evaluation, reports, cross-validation, ensembles, persistence, and progress callbacks.

The examples use small synthetic insurance-style frequency and severity data so the notebook can run without external files. Optional backends such as LightGBM, XGBoost, CatBoost, SHAP, and UMAP require their corresponding extras to be installed.

## 1. Imports and Shared Settings

The public API is Polars-native. Numpy conversion happens inside model wrappers.

In [ ]:
import os
from pathlib import Path

import numpy as np
import polars as pl

from ins_gbm.data.loader import load_model_data
from ins_gbm.data.model_data import ModelData, slice_model_data
from ins_gbm.data.schema import FeatureSchema, infer_schema
from ins_gbm.ensemble.blending import BlendingEnsemble
from ins_gbm.ensemble.pipeline import EnsemblePipeline
from ins_gbm.ensemble.stacking import StackingEnsemble
from ins_gbm.evaluation.comparison import compare_reports
from ins_gbm.evaluation.cv_report import CrossValidationReport
from ins_gbm.evaluation.metrics import (
    compute_metrics,
    gamma_deviance,
    mae,
    normalized_gini,
    poisson_deviance,
    rmse,
)
from ins_gbm.evaluation.plots import plot_loss_ratio
from ins_gbm.evaluation.report import EvaluationReport
from ins_gbm.models.catboost import CatBoostModel
from ins_gbm.models.lightgbm import LightGBMModel
from ins_gbm.models.random_forest import RandomForestModel
from ins_gbm.models.xgboost import XGBoostModel
from ins_gbm.persistence.io import load_pipeline, save_pipeline
from ins_gbm.pipeline import ModelPipeline, ModelRecipe
from ins_gbm.preprocessing.encoder import OneHotEncoder, _MISSING_LEVEL, _NUMERIC_FILL
from ins_gbm.preprocessing.pca import PCAReducer
from ins_gbm.preprocessing.pls import PLSReducer
from ins_gbm.preprocessing.steps import PreprocessingStep
from ins_gbm.preprocessing.umap import UMAPReducer
from ins_gbm.progress import PipelineCancelled, ProgressEvent
from ins_gbm.selection.boruta import BorutaSelector
from ins_gbm.selection.importance import ImportancePruner
from ins_gbm.tuning.search_spaces import narrow_search_space
from ins_gbm.tuning.tuner import HyperparameterTuner

OUTPUT_DIR = Path("output/example_usage")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIGDIR = OUTPUT_DIR / "matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

NUMERIC_MISSING = _NUMERIC_FILL     # -999999999.0
CAT_MISSING = _MISSING_LEVEL        # "-999999999"

# Keep examples fast. Increase these values for real modeling work.
SMALL_RF_PARAMS = {
    "n_estimators": 40,
    "max_depth": 6,
    "min_samples_leaf": 5,
    "random_state": 42,
}


## 2. Missing-Value Convention

Before calling `ins_gbm`, fill missing values explicitly:

- Numeric and ordinal columns: `-999999999.0`
- Categorical columns: `"-999999999"`

`OneHotEncoder` treats the categorical sentinel as a normal level. LightGBM and CatBoost convert the numeric sentinel back to NaN at fit time, XGBoost declares it as `missing`, and Random Forest receives it as a real value.

In [ ]:
def apply_missing_value_convention(df: pl.DataFrame, schema: FeatureSchema) -> pl.DataFrame:
    """Fill nulls with the sentinels expected by ins_gbm."""
    expressions = []

    for col in [*schema.numeric, *schema.ordinal]:
        expressions.append(pl.col(col).fill_null(NUMERIC_MISSING))

    for col in schema.categorical:
        expressions.append(pl.col(col).cast(pl.Utf8).fill_null(CAT_MISSING))

    return df.with_columns(expressions)


## 3. Build Example Data

The frequency data uses a Poisson target with exposure. The severity data uses a positive Gamma-style target with observation weights.

In [ ]:
rng = np.random.default_rng(42)

# Frequency data: expected claim count = rate * exposure.
n_frequency = 180
x1 = rng.normal(size=n_frequency)
x3 = rng.gamma(shape=2.0, scale=1.0, size=n_frequency)
x2 = rng.choice(["A", "B", "C"], size=n_frequency, p=[0.50, 0.30, 0.20])
territory = rng.choice(["north", "south"], size=n_frequency)
exposure = rng.uniform(0.5, 2.0, size=n_frequency)

linear_rate = (
    -1.20
    + 0.35 * x1
    - 0.08 * x3
    + 0.25 * (x2 == "B")
    + 0.15 * (territory == "north")
)
claim_rate = np.exp(linear_rate)
claim_count = rng.poisson(claim_rate * exposure)

# A positive benchmark prediction can be used for double-lift or CV comparison.
benchmark_prediction = np.maximum(exposure * np.exp(-1.15 + 0.22 * x1), 1e-9)
cv_fold = np.arange(n_frequency) % 3

# Inject nulls after target generation so we can demonstrate sentinel filling.
x1_with_nulls = x1.astype(object).tolist()
x2_with_nulls = x2.astype(object).tolist()
for idx in range(0, n_frequency, 37):
    x1_with_nulls[idx] = None
for idx in range(5, n_frequency, 41):
    x2_with_nulls[idx] = None

frequency_raw = pl.DataFrame(
    {
        "x1": x1_with_nulls,
        "x2": x2_with_nulls,
        "x3": x3,
        "territory": territory,
        "exposure": exposure,
        "claim_count": claim_count,
        "benchmark_prediction": benchmark_prediction,
        "cv_fold": cv_fold,
    }
)

frequency_schema = FeatureSchema(
    numeric=["x1", "x3"],
    categorical=["x2", "territory"],
)
frequency_prepared = apply_missing_value_convention(frequency_raw, frequency_schema)
frequency_path = OUTPUT_DIR / "frequency.parquet"
frequency_prepared.write_parquet(frequency_path)

frequency_prepared.head()


In [ ]:
# Severity data: strictly positive target plus optional sample weight.
n_severity = 160
s1 = rng.normal(size=n_severity)
s2 = rng.choice(["low", "medium", "high"], size=n_severity, p=[0.45, 0.40, 0.15])
s3 = rng.uniform(0.0, 1.0, size=n_severity)
weight = rng.uniform(0.5, 1.5, size=n_severity)

mean_severity = np.exp(2.50 + 0.20 * s1 + 0.30 * (s2 == "high") - 0.10 * s3)
severity = np.maximum(mean_severity * rng.gamma(shape=2.0, scale=0.5, size=n_severity), 1e-6)
benchmark_severity = np.maximum(np.exp(2.45 + 0.12 * s1), 1e-6)

s1_with_nulls = s1.astype(object).tolist()
s2_with_nulls = s2.astype(object).tolist()
for idx in range(3, n_severity, 39):
    s1_with_nulls[idx] = None
for idx in range(7, n_severity, 43):
    s2_with_nulls[idx] = None

severity_raw = pl.DataFrame(
    {
        "s1": s1_with_nulls,
        "s2": s2_with_nulls,
        "s3": s3,
        "severity": severity,
        "weight": weight,
        "benchmark_severity": benchmark_severity,
    }
)

severity_schema = FeatureSchema(
    numeric=["s1", "s3"],
    categorical=["s2"],
)
severity_prepared = apply_missing_value_convention(severity_raw, severity_schema)
severity_path = OUTPUT_DIR / "severity.parquet"
severity_prepared.write_parquet(severity_path)

severity_prepared.head()


## 4. Load Data and Define Schemas

`load_model_data` reads Parquet and returns a validated `ModelData`. You can supply a `FeatureSchema` explicitly or infer one from Polars dtypes.

In [ ]:
frequency_data = load_model_data(
    path=str(frequency_path),
    target="claim_count",
    exposure="exposure",
    feature_cols=["x1", "x2", "x3", "territory"],
    schema=frequency_schema,
    objective="poisson",
    cv_fold="cv_fold",
    comparison_cols=["benchmark_prediction"],
)

severity_data = load_model_data(
    path=str(severity_path),
    target="severity",
    weight="weight",
    feature_cols=["s1", "s2", "s3"],
    schema=severity_schema,
    objective="gamma",
    comparison_cols=["benchmark_severity"],
)

inferred_frequency_schema = infer_schema(frequency_prepared, ["x1", "x2", "x3", "territory"])

print(f"frequency rows: {frequency_data.n_rows}")
print(f"frequency features: {frequency_data.feature_names}")
print(f"inferred feature order: {inferred_frequency_schema.all_features()}")


## 5. Explicit Holdout, Slice, and Add Offsets

The caller owns final holdouts. `slice_model_data` is useful for constructing a deterministic example holdout or custom folds. `with_offset` adds link-scale offsets, such as a base model score, without mutating the original data.

In [ ]:
frequency_holdout = slice_model_data(frequency_data, np.arange(100))
first_five_rows = slice_model_data(frequency_data, np.arange(5))

# Offset example: convert benchmark claim count to a link-scale rate offset.
benchmark_rate = frequency_data.comparisons["benchmark_prediction"] / frequency_data.exposure
frequency_with_offset = frequency_data.with_offset(
    pl.Series("offset", np.log(np.maximum(benchmark_rate.to_numpy(), 1e-9)))
)

print(f"model-data rows: {frequency_data.n_rows}")
print(f"holdout rows: {frequency_holdout.n_rows}")
print(f"sliced rows: {first_five_rows.n_rows}")
print(f"offset attached: {frequency_with_offset.offset is not None}")


## 6. Encoding and Dimensionality Reduction

`OneHotEncoder` fits on training features and keeps a stable output column order. Reducers transform encoded numeric matrices. PLS is supervised and must receive the training target.

In [ ]:
fitted_encoder = OneHotEncoder().fit(frequency_data.features, frequency_schema)
encoded_frequency = frequency_data.with_features(fitted_encoder.transform(frequency_data.features))
encoded_holdout = frequency_holdout.with_features(fitted_encoder.transform(frequency_holdout.features))

print(fitted_encoder.output_feature_names())
encoded_frequency.features.head()


In [ ]:
pca = PCAReducer(n_components=2).fit(encoded_frequency.features)
pca_train = pca.transform(encoded_frequency.features)

pls = PLSReducer(n_components=2).fit(encoded_frequency.features, encoded_frequency.target)
pls_train = pls.transform(encoded_frequency.features)

# UMAP is optional. Install `umap-learn` to run this reducer.
try:
    umap_reducer = UMAPReducer(n_components=2, n_neighbors=10, seed=42).fit(encoded_frequency.features)
    umap_train = umap_reducer.transform(encoded_frequency.features)
except ImportError as exc:
    umap_train = pl.DataFrame({"status": [f"UMAP skipped: {exc}"]})

pl.DataFrame(
    {
        "pca_columns": [pca.output_feature_names()],
        "pls_columns": [pls.output_feature_names()],
        "umap_preview_columns": [umap_train.columns],
    }
)


## 7. Model Backends and Capabilities

Every model wrapper implements `fit`, `default_search_space`, and `capabilities`. Optional backends are skipped if their runtime dependency is not installed.

In [ ]:
backend_configs = {
    "random_forest": (RandomForestModel, SMALL_RF_PARAMS),
    "lightgbm": (LightGBMModel, {"n_estimators": 30, "verbose": -1}),
    "xgboost": (XGBoostModel, {"n_estimators": 30, "verbosity": 0}),
    "catboost": (CatBoostModel, {"iterations": 30, "verbose": 0, "allow_writing_files": False}),
}

backend_rows = []
for backend_name, (model_cls, params) in backend_configs.items():
    model = model_cls(objective="poisson")
    caps = model.capabilities()
    row = {
        "backend": backend_name,
        "poisson": caps.supports_poisson,
        "gamma": caps.supports_gamma,
        "offset": caps.supports_offset,
        "sample_weight": caps.supports_sample_weight,
        "feature_importance": caps.supports_feature_importance,
    }
    try:
        fitted = model.fit(encoded_frequency, params=params)
        prediction = fitted.predict(encoded_holdout, prediction_type="response")[0]
        row["fit_status"] = "ok"
        row["first_prediction"] = float(prediction)
    except Exception as exc:
        row["fit_status"] = f"skipped: {type(exc).__name__}: {exc}"
        row["first_prediction"] = None
    backend_rows.append(row)

pl.DataFrame(backend_rows)


## 8. Full Frequency Pipeline

`ModelPipeline` executes optional cross-validation tuning, encoder, selector, preprocessors, full-data model fit, and metadata capture. Evaluate an explicit holdout after fitting.

In [ ]:
frequency_recipe = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    params=SMALL_RF_PARAMS,
)

frequency_result = ModelPipeline(
    data=frequency_data,
    recipe=frequency_recipe,
).run()

frequency_report = frequency_result.evaluate(frequency_holdout)
frequency_report.metrics()


## 9. Predict on Raw Data

`FittedPipeline.predict` accepts raw `ModelData` and applies the fitted transform chain. `predict_raw` accepts a feature `DataFrame` plus exposure or weight when needed.

In [ ]:
frequency_predictions = frequency_result.predict(
    frequency_holdout,
    prediction_type="response",
)
frequency_rates = frequency_result.predict(
    frequency_holdout,
    prediction_type="rate",
)
raw_feature_predictions = frequency_result.predict_raw(
    frequency_holdout.features.head(5),
    exposure=frequency_holdout.exposure.head(5),
    prediction_type="response",
)

pl.DataFrame(
    {
        "actual_count": frequency_holdout.target.head(5),
        "predicted_count": frequency_predictions.head(5),
        "predicted_rate": frequency_rates.head(5),
        "predict_raw_count": raw_feature_predictions,
    }
)


## 10. Full Severity Pipeline

Gamma severity requires strictly positive targets. For Gamma models, `prediction_type="rate"` is not valid.

In [ ]:
severity_recipe = ModelRecipe(
    model=RandomForestModel(objective="gamma"),
    encoder=OneHotEncoder(),
    params=SMALL_RF_PARAMS,
)

severity_result = ModelPipeline(
    data=severity_data,
    recipe=severity_recipe,
).run()

severity_holdout = slice_model_data(severity_data, np.arange(100))
severity_report = severity_result.evaluate(severity_holdout)
severity_report.metrics()


## 11. Hyperparameter Tuning and Search Spaces

`HyperparameterTuner` uses Optuna. It refits the encoder, selector, and preprocessor inside each CV fold to avoid leakage. Keep `n_trials` low for examples and increase it for production work.

In [ ]:
tuning_recipe = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    tuning=HyperparameterTuner(
        n_trials=2,
        cv_folds=2,
        metric="poisson_deviance",
        seed=7,
    ),
)

tuned_result = ModelPipeline(
    data=frequency_data,
    recipe=tuning_recipe,
).run()

tuned_result.tuning_history


In [ ]:
# The tuner can also use predefined folds from ModelData.cv_fold.
fold_tuner = HyperparameterTuner(
    n_trials=2,
    metric="poisson_deviance",
    use_data_folds=True,
    seed=7,
)
best_params, fold_history = fold_tuner.tune(
    data=frequency_data,
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    schema=frequency_schema,
)

# narrow_search_space is a helper for replacing selected Optuna distributions.
# A custom model wrapper could return this restricted space from default_search_space().
import optuna

restricted_space = narrow_search_space(
    RandomForestModel(objective="poisson").default_search_space(),
    n_estimators=optuna.distributions.IntDistribution(20, 80),
)

print(best_params)
print(restricted_space["n_estimators"])
fold_history


## 12. Feature Selection

`BorutaSelector` can be used inside `ModelPipeline`. `ImportancePruner` is a manual post-fit utility: fit a model first, then prune based on that fitted model's feature importance.

In [ ]:
selection_recipe = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    selection=BorutaSelector(base_estimator="random_forest", max_iter=3, seed=42),
    params=SMALL_RF_PARAMS,
)

selection_result = ModelPipeline(
    data=frequency_data,
    recipe=selection_recipe,
).run()

manual_fitted_model = RandomForestModel(objective="poisson").fit(
    encoded_frequency,
    params=SMALL_RF_PARAMS,
)
importance_table = manual_fitted_model.feature_importance()
importance_selection = ImportancePruner(top_n=3).fit(encoded_frequency, manual_fitted_model)

print(f"Boruta-selected features: {selection_result.selected_features}")
print(f"ImportancePruner-selected features: {importance_selection.selected_features()}")
importance_table.sort("importance", descending=True)


## 13. Evaluation Metrics and Plots

Use the report object for standard metrics and plots, or call the metric and plotting functions directly.

In [ ]:
test_predictions = frequency_result.fitted_model.predict(
    frequency_report.evaluation_data,
    prediction_type="response",
)

metric_table = compute_metrics(
    objective="poisson",
    actual=frequency_report.evaluation_data.target,
    predicted=test_predictions,
    exposure=frequency_report.evaluation_data.exposure,
)

single_metric_values = {
    "poisson_deviance": poisson_deviance(
        frequency_report.evaluation_data.target,
        test_predictions,
        weights=frequency_report.evaluation_data.exposure,
    ),
    "gini": normalized_gini(
        frequency_report.evaluation_data.target,
        test_predictions,
        weights=frequency_report.evaluation_data.exposure,
    ),
    "rmse": rmse(frequency_report.evaluation_data.target, test_predictions),
    "mae": mae(frequency_report.evaluation_data.target, test_predictions),
}

metric_table


In [ ]:
# In a notebook, the returned matplotlib Figure objects display automatically.
fig_lift = frequency_report.plot_lift()
fig_ave = frequency_report.plot_ave()
fig_calibration = frequency_report.plot_calibration()
fig_importance = frequency_report.plot_feature_importance()
fig_double_lift = frequency_report.plot_double_lift("benchmark_prediction")

# Loss-ratio plotting needs loss and premium series supplied by the caller.
loss = frequency_report.evaluation_data.target * 500.0
premium = test_predictions * 550.0 + 100.0
fig_loss_ratio = plot_loss_ratio(
    actual=frequency_report.evaluation_data.target,
    predicted=test_predictions,
    loss=loss,
    premium=premium,
)


## 14. Reports, Comparisons, and Cross-Validation

`EvaluationReport.compare` compares fitted models. `compare_reports` compares `EvaluationReport` and `CVResult` objects side by side. This fit also shows per-run raw feature selection and PCA applied only to numeric columns while encoded categorical columns pass through.

In [ ]:
pca_recipe = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    preprocessing=[
        PreprocessingStep(
            name="numeric_pca",
            preprocessor=PCAReducer(n_components=2),
            feature_names=["x1", "x3"],
        ),
    ],
    params=SMALL_RF_PARAMS,
)
pca_result = ModelPipeline(
    data=frequency_data,
    recipe=pca_recipe,
).run(feature_names=["x1", "x2", "x3"])

comparison_report = EvaluationReport.compare(
    models={
        "rf": (frequency_result.fitted_model, frequency_result.train_data, frequency_report.evaluation_data),
        "rf_subset_pca": (pca_result.fitted_model, pca_result.train_data, pca_result.evaluate(frequency_holdout).evaluation_data),
    },
)

side_by_side = compare_reports(
    {
        "rf": frequency_report,
        "rf_subset_pca": pca_result.evaluate(frequency_holdout),
    }
)

print(comparison_report.metrics())
side_by_side


## 15. Multiple Targeted Preprocessors

Pass multiple `PreprocessingStep` objects through `ModelRecipe.preprocessing` to reduce separate feature groups while retaining all other encoded columns. Steps are fit and applied in list order; each is refit on the training partition inside cross-validation and tuning.


In [ ]:
multi_preprocessor_recipe = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    encoder=OneHotEncoder(),
    preprocessing=[
        PreprocessingStep(
            name="x1_pca",
            preprocessor=PCAReducer(n_components=1),
            feature_names=["x1"],
        ),
        PreprocessingStep(
            name="x3_pca",
            preprocessor=PCAReducer(n_components=1),
            feature_names=["x3"],
        ),
    ],
    params=SMALL_RF_PARAMS,
)
multi_preprocessor_result = ModelPipeline(
    data=frequency_data,
    recipe=multi_preprocessor_recipe,
).run(feature_names=["x1", "x2", "x3", "territory"])

# Both reducer outputs replace their respective source columns; encoded
# categorical columns pass through unchanged.
multi_preprocessor_result.train_data.features.head(), [
    step.output_feature_names() for step in multi_preprocessor_result.preprocessors
]


In [ ]:
# CrossValidationReport expects fold and benchmark columns inside features,
# then drops them before fitting the recipe.
cv_features = frequency_prepared.select(
    ["x1", "x2", "x3", "territory", "benchmark_prediction", "cv_fold"]
)
cv_schema = FeatureSchema(
    numeric=["x1", "x3", "benchmark_prediction"],
    categorical=["x2", "territory"],
    passthrough=["cv_fold"],
)
cv_data = ModelData(
    features=cv_features,
    target=frequency_prepared["claim_count"],
    exposure=frequency_prepared["exposure"],
    weight=None,
    feature_names=list(cv_features.columns),
    schema=cv_schema,
    objective="poisson",
).validate()

cv_result = CrossValidationReport(
    recipe=frequency_recipe,
    data=cv_data,
    n_folds=3,
    benchmark_col="benchmark_prediction",
    fold_col="cv_fold",
).run()

compare_reports({"holdout": frequency_report, "cross_validation": cv_result})


## 16. Ensembles

Blending and stacking combine pre-fitted pipelines. The example below uses numeric-only pipelines so the ensemble report can score the transformed test data directly.

In [ ]:
numeric_frequency_data = ModelData(
    features=frequency_data.features.select(["x1", "x3"]),
    target=frequency_data.target,
    exposure=frequency_data.exposure,
    weight=None,
    feature_names=["x1", "x3"],
    schema=FeatureSchema(numeric=["x1", "x3"], categorical=[]),
    objective="poisson",
).validate()

numeric_recipe_a = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    params={**SMALL_RF_PARAMS, "random_state": 11},
)
numeric_recipe_b = ModelRecipe(
    model=RandomForestModel(objective="poisson"),
    params={**SMALL_RF_PARAMS, "max_depth": 4, "random_state": 22},
)

numeric_result_a = ModelPipeline(
    data=numeric_frequency_data,
    recipe=numeric_recipe_a,
).run()
numeric_result_b = ModelPipeline(
    data=numeric_frequency_data,
    recipe=numeric_recipe_b,
).run()

fixed_blend = BlendingEnsemble(mode="fixed", weights=[0.60, 0.40]).fit(
    [numeric_result_a, numeric_result_b]
)
validation_blend = BlendingEnsemble(mode="validation").fit(
    [numeric_result_a, numeric_result_b],
    validation_data=numeric_result_a.train_data,
)
stacked = StackingEnsemble(cv_folds=2, seed=42).fit(
    [numeric_result_a, numeric_result_b]
)
ensemble_result = EnsemblePipeline(
    fitted_pipelines=[numeric_result_a, numeric_result_b],
    method="blending",
    blend_mode="fixed",
    blend_weights=[0.60, 0.40],
).run()

pl.DataFrame(
    {
        "fixed_blend_head": fixed_blend.predict(numeric_frequency_data).head(5),
        "validation_weights": [validation_blend.weights[:5]] * 5,
        "stacked_head": stacked.predict(numeric_frequency_data).head(5),
        "pipeline_blend_head": ensemble_result.predict(numeric_frequency_data).head(5),
    }
)


## 17. Persistence, Exports, and Metadata

`save_pipeline` uses cloudpickle so local prediction closures survive serialization. The save directory includes the pipeline artifact, metadata, metrics, and tuning history when present.

In [ ]:
pipeline_dir = OUTPUT_DIR / "frequency_pipeline"
report_dir = OUTPUT_DIR / "frequency_report"
frequency_report.export(str(report_dir))

try:
    import cloudpickle  # save_pipeline requires cloudpickle at runtime.

    save_pipeline(frequency_result, str(pipeline_dir))
    loaded_frequency = load_pipeline(str(pipeline_dir))
    loaded_predictions = loaded_frequency.predict(
        frequency_holdout,
        prediction_type="response",
    ).head(5)
    pipeline_files = sorted(path.name for path in pipeline_dir.iterdir())
except ImportError as exc:
    loaded_predictions = pl.Series("persistence_status", [f"save/load skipped: {exc}"])
    pipeline_files = []

print(f"pipeline files: {pipeline_files}")
print(f"report files: {sorted(path.name for path in report_dir.iterdir())}")
print(f"metadata objective: {frequency_result.metadata.objective}")
loaded_predictions


## 18. Progress Callbacks and Cancellation

`ModelPipeline` and `HyperparameterTuner` can emit `ProgressEvent` objects. `should_stop` can cancel a run by raising `PipelineCancelled`.

In [ ]:
events: list[ProgressEvent] = []

def record_progress(event: ProgressEvent) -> None:
    events.append(event)

progress_result = ModelPipeline(
    data=frequency_data,
    recipe=frequency_recipe,
    progress=record_progress,
    should_stop=lambda: False,
).run()

try:
    ModelPipeline(
        data=frequency_data,
        recipe=frequency_recipe,
        should_stop=lambda: True,
    ).run()
except PipelineCancelled as exc:
    cancelled_message = str(exc)

pl.DataFrame(
    {
        "stage": [event.stage for event in events],
        "message": [event.message for event in events],
        "current": [event.current for event in events],
        "total": [event.total for event in events],
    }
).with_columns(pl.lit(cancelled_message).alias("cancelled_example"))


## 19. What to Change for Real Data

Replace the synthetic Parquet files with your prepared data, keep the sentinel missing-value convention, choose the model backend that matches your installed extras, increase tuning trials and CV folds, and export reports and saved pipelines to a durable model artifact location.